<a href="https://colab.research.google.com/github/nxxhhaaa/Healthcare/blob/main/hospital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Healthcare Appointment Data Analysis

The aim of this project is to analyse healthcare appointment data to understand patient waiting times, appointment patterns, hospital performance, and patient satisfaction.

Python is used to clean and prepare the dataset before visualising the results in Power BI.

The dataset contains information about appointments, patients, hospitals, waiting times, consultation durations, satisfaction scores, and costs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Loading and Understanding the Data

The pandas library is imported to help load, organise, and analyse the dataset.

The CSV file is loaded into a DataFrame called `df`. A DataFrame is a table containing rows and columns.


In [ ]:
url = "https://raw.githubusercontent.com/nxxhhaaa/Healthcare/main/healthcare_appointments_sql_ready.csv"
df = pd.read_csv(url)


In [ ]:
df.head()

,appointment_id,patient_id,hospital,department,appointment_datetime,arrival_datetime,discharge_datetime,age,gender,appointment_status,waiting_time_minutes,consultation_duration_minutes,referral_source,postcode_area,patient_satisfaction,follow_up_required,cost_gbp,data_source
0,A00001,P10377,Royal Victoria Hospital,Outpatient,16/10/2025,16/10/2025,16/10/2025,75.0,F,Pending,75.0,29.0,Emergency,BT9,6.0,Yes,196.82,Legacy DB
1,A00002,P10110,RVH,Cardiology,01/07/2024,07/01/2024,NaN,NaN,F,Completed,111.0,27.0,Online,NaN,1.0,Yes,126.27,Legacy DB
2,A00003,P10437,Ulster Hospital,Neurology,03/12/2024,12/03/2024,12/03/2024,46.0,Prefer not to say,D,113.0,45.0,Walk-in,BT9,6.0,No,107.44,ClinicSystem
3,A00004,P10117,Belfast City Hospital,Pediatrics,10-17-2025,17/10/2025,17/10/2025,NaN,Prefer not to say,D,76.0,23.0,Specialist,BT9,4.0,Yes,107.74,Legacy DB
4,A00005,P10386,Belfast City Hospital,Accident & Emergency,20/08/2025,20/08/2025,20/08/2025,145.0,M,D,66.0,5.0,NaN,BT7,6.0,Yes,7.01,HIS


The `head()` function displays the first five rows to help us understand the structure of the data.

In [ ]:
df.shape

(1801, 18)

The shape attribute shows the total number of rows and columns, while info() provides information about column names, data types, and non-missing values.

In [ ]:
df.describe()

,age,waiting_time_minutes,consultation_duration_minutes,patient_satisfaction,cost_gbp
count,891.000000,1785.000000,1797.000000,1223.000000,1668.000000
mean,79.262626,68.625210,34.316639,2.995094,146.807326
std,50.935325,120.875344,37.075743,2.370711,76.913045
min,0.000000,0.000000,-10.000000,0.000000,0.000000
25%,33.000000,36.000000,19.000000,0.000000,91.572500
50%,71.000000,58.000000,32.000000,3.000000,145.320000
75%,145.000000,79.000000,44.000000,6.000000,198.587500
max,145.000000,1440.000000,600.000000,6.000000,459.100000


In [ ]:
df.isnull().sum()

,0
appointment_id,0
patient_id,0
hospital,224
department,0
appointment_datetime,23
arrival_datetime,110
discharge_datetime,212
age,910
gender,349
appointment_status,201


The isnull().sum() function identifies the number of missing values in each column.

In [ ]:
df.duplicated().sum()

np.int64(0)

## 2. Identifying Duplicate Records

Duplicate records can affect the accuracy of data analysis.

The `duplicated()` function identifies repeated rows, and `sum()` counts how many duplicates exist.

In [ ]:
# check unusual values
invalid = df[(df["age"]<0)| (df["age"]>110)]
display(invalid)

,appointment_id,patient_id,hospital,department,appointment_datetime,arrival_datetime,discharge_datetime,age,gender,appointment_status,waiting_time_minutes,consultation_duration_minutes,referral_source,postcode_area,patient_satisfaction,follow_up_required,cost_gbp,data_source
4,A00005,P10386,Belfast City Hospital,Accident & Emergency,20/08/2025,20/08/2025,20/08/2025,145.0,M,D,66.0,5.0,NaN,BT7,6.0,Yes,7.01,HIS
5,A00006,P10232,Belfast City Hospital,Out Patient,30/09/2024,30/09/2024,30/09/2024,145.0,M,Cancelled,38.0,13.0,GP,NaN,2.0,Yes,170.14,Legacy DB
7,A00008,P10264,NaN,CARDIOLOGY,12/07/2025,07/12/2025,NaN,145.0,M,Pending,50.0,33.0,Specialist,BT1,1.0,Yes,31.02,LegacyDB
20,A00021,P10285,Ulster Hospital,Accident & Emergency,06/01/2024,06/01/2024,06/01/2024,145.0,NaN,D,69.0,49.0,Walk-in,BT9,0.0,NaN,106.67,HIS
22,A00023,P10194,NaN,CARDIOLOGY,01/03/2025,NaN,01/03/2025,145.0,NaN,Completed,86.0,56.0,Specialist,BT9,0.0,No,NaN,LegacyDB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1774,A01789,P10160,RVH,Orthopedics,05/02/2024,NaN,05/02/2024,145.0,F,Cancelled,49.0,37.0,GP,BT12,NaN,No,310.46,Legacy DB
1778,A01793,P10216,Belfast City Hospital,Accident & Emergency,05-25-2024 11:30,25/05/2024,25/05/2024,145.0,Prefer not to say,Completed,57.0,21.0,Emergency,BT2,1.0,No,20.30,HIS
1783,A01798,P10231,Belfast City Hospital,GP Referral,20/11/2025,20/11/2025,20/11/2025,145.0,M,Pending,128.0,54.0,Online,BT2,6.0,Yes,82.68,HIS
1789,A01381,P10271,RVH,Pediatrics,28/05/2024,28/05/2024,28/05/2024,145.0,M,D,50.0,31.0,GP,BT9,6.0,Yes,89.89,Legacy DB


In [ ]:
display(df[(df["consultation_duration_minutes"]<0)])

,appointment_id,patient_id,hospital,department,appointment_datetime,arrival_datetime,discharge_datetime,age,gender,appointment_status,waiting_time_minutes,consultation_duration_minutes,referral_source,postcode_area,patient_satisfaction,follow_up_required,cost_gbp,data_source
1000,A01009,P10113,Belfast City Hospital,Accident & Emergency,02/03/2025,02/03/2025,02/03/2025,145.0,NaN,D,0.0,-10.0,Emergency,BT2,NaN,Yes,207.19,ClinicSystem
1231,A01241,P10292,Belfast City Hospital,GP Referral,10/11/2024,10/11/2024,10/11/2024,68.0,Non-binary,Completed,35.0,-10.0,Self-referral,BT12,0.0,Yes,94.26,HIS
1428,A01440,P10019,Mater Hospital,Accident & Emergency,25/02/2024,25/02/2024,25/02/2024,48.0,M,Discharged,39.0,-10.0,GP,NaN,NaN,NaN,0.00,LegacyDB


In [ ]:
date_col = [
    "appointment_datetime",
    "arrival_datetime",
    "discharge_datetime"
]
for col in date_col:
    print(f"\n{col}")
    print("Missing values:", df[col].isna().sum())
    print("Sample values:", df[col].dropna().head(5).tolist())



appointment_datetime
Missing values: 23
Sample values: ['16/10/2025', '01/07/2024', '03/12/2024', '10-17-2025 ', '20/08/2025']

arrival_datetime
Missing values: 110
Sample values: ['16/10/2025', '07/01/2024', '12/03/2024', '17/10/2025', '20/08/2025']

discharge_datetime
Missing values: 212
Sample values: ['16/10/2025', '12/03/2024', '17/10/2025', '20/08/2025', '30/09/2024']


In [ ]:
print("unique id : " , df['appointment_id'].nunique())
print("duplicates :" , df['appointment_id'].duplicated().sum())
repeated_id = df[df['appointment_id'].duplicated(keep=False)]
display(repeated_id.head(10))


unique id :  1783
duplicates : 18


,appointment_id,patient_id,hospital,department,appointment_datetime,arrival_datetime,discharge_datetime,age,gender,appointment_status,waiting_time_minutes,consultation_duration_minutes,referral_source,postcode_area,patient_satisfaction,follow_up_required,cost_gbp,data_source
56,A00057,P10410,NaN,Emergency,03/03/2025,03/03/2025,03/03/2025,13.0,F,Completed,91.0,39.0,NaN,NaN,NaN,No,210.76,LegacyDB
119,A00057,P10052,Mater Hospital,Cardiology,27/06/2025,27/06/2025,27/06/2025,9.0,M,Discharged,124.0,27.0,Online,NaN,NaN,Yes,84.70,Legacy DB
233,A00184,P10134,Mater Hospital,Oncology,NaN,10/02/2024,10/02/2024,145.0,Non-binary,Cancelled,44.0,50.0,NaN,BT9,NaN,NaN,98.85,HIS
234,A00184,P10076,Mater Hospital,A&E,05-22-2024 14:10,22/05/2024,22/05/2024,84.0,F,D,87.0,11.0,Walk-in,BT2,0.0,No,187.38,HIS
285,A00237,P10291,Belfast City Hospital,Cardiology,27/11/2024,27/11/2024,27/11/2024,NaN,M,Discharged,79.0,42.0,GP,BT7,NaN,No,120.24,Legacy DB
342,A00237,P10295,RVH,Orthopedics,27/02/2024,27/02/2024,27/02/2024,NaN,F,Discharged,69.0,9.0,Walk-in,BT7,6.0,Yes,218.92,Legacy DB
363,A00289,P10437,Mater Hospital,Orthopedics,15/11/2024,15/11/2024,15/11/2024,NaN,F,D,71.0,34.0,GP,BT9,0.0,Yes,184.40,HIS
377,A00289,P10102,Mater Hospital,Cardiology,14/04/2024,14/04/2024,14/04/2024,NaN,NaN,Cancelled,68.0,76.0,GP,BT7,NaN,Yes,93.98,HIS
379,A00347,P10246,NaN,Out Patient,08/09/2025,09/08/2025,09/08/2025,NaN,NaN,NaN,33.0,63.0,NaN,NaN,6.0,Yes,160.83,ClinicSystem
380,A00347,P10419,Belfast City Hospital,Emergency,09/01/2024,09/01/2024,09/01/2024,81.0,Prefer not to say,No-show,35.0,23.0,Specialist,BT9,NaN,Yes,119.37,Legacy DB


A separate check is performed on `appointment_id` to identify repeated appointment identifiers. This is important because two records may have the same appointment ID without being completely identical.

In [ ]:
missing_summary = pd.DataFrame({
    "missing count : " : df.isnull().sum(),
    " percentage :" : (df.isnull().sum()/len(df)*100).round(2)
})
display(missing_summary)


,missing count :,percentage :
appointment_id,0,0.00
patient_id,0,0.00
hospital,224,12.44
department,0,0.00
appointment_datetime,23,1.28
arrival_datetime,110,6.11
discharge_datetime,212,11.77
age,910,50.53
gender,349,19.38
appointment_status,201,11.16


## 3. Missing-Value Analysis

Missing values occur when information has not been recorded or is unavailable.

The dataset is checked to identify missing values in each column and calculate their percentage.

A missing-value report is created using pandas. The report includes both the number of missing values and the percentage of missing values for every column.

This step helps identify data-quality problems and determine which columns may require further investigation.

Age has approximately 50.53% missing values, while patient satisfaction has approximately 32.09% missing values in the initial dataset. These limitations need to be considered when interpreting the results.

In [ ]:
df_clean = df.copy()
display(df_clean)

categorical_col = [
    "gender",
    "appointment_status",
    "referral_source",
    "follow_up_required",
    "postcode_area"
]
for col in categorical_col:
  df_clean[col]= (df_clean[col]
    .fillna("Unknown")
    .astype("str").str.strip())

,appointment_id,patient_id,hospital,department,appointment_datetime,arrival_datetime,discharge_datetime,age,gender,appointment_status,waiting_time_minutes,consultation_duration_minutes,referral_source,postcode_area,patient_satisfaction,follow_up_required,cost_gbp,data_source
0,A00001,P10377,Royal Victoria Hospital,Outpatient,16/10/2025,16/10/2025,16/10/2025,75.0,F,Pending,75.0,29.0,Emergency,BT9,6.0,Yes,196.82,Legacy DB
1,A00002,P10110,RVH,Cardiology,01/07/2024,07/01/2024,NaN,NaN,F,Completed,111.0,27.0,Online,NaN,1.0,Yes,126.27,Legacy DB
2,A00003,P10437,Ulster Hospital,Neurology,03/12/2024,12/03/2024,12/03/2024,46.0,Prefer not to say,D,113.0,45.0,Walk-in,BT9,6.0,No,107.44,ClinicSystem
3,A00004,P10117,Belfast City Hospital,Pediatrics,10-17-2025,17/10/2025,17/10/2025,NaN,Prefer not to say,D,76.0,23.0,Specialist,BT9,4.0,Yes,107.74,Legacy DB
4,A00005,P10386,Belfast City Hospital,Accident & Emergency,20/08/2025,20/08/2025,20/08/2025,145.0,M,D,66.0,5.0,NaN,BT7,6.0,Yes,7.01,HIS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1796,A01717,P10251,Belfast City Hospital,Oncology,12/10/2025,10/12/2025,10/12/2025,85.0,F,No-show,51.0,5.0,Self-referral,NaN,NaN,No,327.17,HIS
1797,A01750,P10330,Ulster Hospital,CARDIOLOGY,05/01/2024,05/01/2024,05/01/2024,NaN,M,D,37.0,5.0,Emergency,BT9,4.0,Yes,210.35,HIS
1798,A01750,P10008,Royal Victoria Hospital,Neurology,12/07/2024,12/07/2024,12/07/2024,20.0,F,Discharged,9.0,24.0,Self-referral,NaN,6.0,No,124.39,Legacy DB
1799,A01759,P10227,Mater Hospital,Oncology,20/04/2025,20/04/2025,NaN,145.0,NaN,Completed,43.0,48.0,GP,BT12,2.0,Yes,154.88,LegacyDB


## 4. Handling Missing Categorical Values

Categorical variables represent groups or labels, such as gender, referral source, postcode area, and follow-up status.

The `fillna()` function is used to replace missing values in selected categorical columns with the label "Unknown".

This approach preserves the records while making it clear that the original information is unavailable.

Unknown values are kept separate from known categories to avoid making unsupported assumptions about patients or appointments.

In [ ]:
numeric_col = [
    "age",
    "waiting_time_minutes",
    "consultation_duration_minutes",
    "patient_satisfaction",
    "cost_gbp"
]
for col in numeric_col:
  df_clean[col]=pd.to_numeric(
   df_clean[col])

print("Numerical data quality report:")

display(df_clean[numeric_col].describe().T)

print("\nInvalid or unusual values:")

print(
    "Age below 0 or above 110:",
    ((df_clean["age"] < 0) | (df_clean["age"] > 110)).sum()
)

print(
    "Negative waiting times:",
    (df_clean["waiting_time_minutes"] < 0).sum()
)

print(
    "Negative consultation durations:",
    (df_clean["consultation_duration_minutes"] < 0).sum()
)

print(
    "Satisfaction scores outside 1–10:",
    (
        (df_clean["patient_satisfaction"] < 1) |
        (df_clean["patient_satisfaction"] > 10)
    ).sum()
)

print(
    "Negative costs:",
    (df_clean["cost_gbp"] < 0).sum()
)


Numerical data quality report:


,count,mean,std,min,25%,50%,75%,max
age,891.0,79.262626,50.935325,0.0,33.0000,71.00,145.0000,145.0
waiting_time_minutes,1785.0,68.625210,120.875344,0.0,36.0000,58.00,79.0000,1440.0
consultation_duration_minutes,1797.0,34.316639,37.075743,-10.0,19.0000,32.00,44.0000,600.0
patient_satisfaction,1223.0,2.995094,2.370711,0.0,0.0000,3.00,6.0000,6.0
cost_gbp,1668.0,146.807326,76.913045,0.0,91.5725,145.32,198.5875,459.1



Invalid or unusual values:
Age below 0 or above 110: 288
Negative waiting times: 0
Negative consultation durations: 3
Satisfaction scores outside 1–10: 309
Negative costs: 0


## 5. Validating Numerical Values

Numerical columns are checked for values that may be invalid or unrealistic.

Age values below zero or above 110 are identified for review. Negative consultation durations are also checked because they do not represent valid elapsed durations.

Invalid values are replaced with missing values rather than being changed to arbitrary numbers.

Waiting times and costs are checked for negative values.

These checks improve data quality and help prevent incorrect conclusions during the analysis.

In [ ]:
invalid_age = (df_clean["age"] < 0) | (df_clean["age"] > 110)
df_clean.loc[invalid_age, "age"] = float("nan")
invalid_duration = (
    df_clean["consultation_duration_minutes"] < 0
)

df_clean.loc[invalid_duration, "consultation_duration_minutes"] = float("nan")

print("Invalid ages remaining:",
      ((df_clean["age"] < 0) | (df_clean["age"] > 110)).sum())

print("Negative consultation durations remaining:",
      (df_clean["consultation_duration_minutes"] < 0).sum())


Invalid ages remaining: 0
Negative consultation durations remaining: 0


In [ ]:

print(
    df_clean["patient_satisfaction"]
    .value_counts(dropna=False)
    .sort_index()
)


patient_satisfaction
0.0    309
1.0    136
2.0    106
3.0    126
4.0    109
5.0    121
6.0    316
NaN    578
Name: count, dtype: int64


In [ ]:
df_clean.loc[df_clean["patient_satisfaction"]==0,"patient_satisfaction"]=float("nan")
print(df_clean["patient_satisfaction"].value_counts(dropna=False).sort_index())


patient_satisfaction
1.0    136
2.0    106
3.0    126
4.0    109
5.0    121
6.0    316
NaN    887
Name: count, dtype: int64


## 6. Patient Satisfaction Data Cleaning

The patient satisfaction column is examined using `value_counts()` to understand the distribution of recorded scores.

The `dropna=False` argument includes missing values in the results, while `sort_index()` displays the scores in numerical order.

Based on the intended satisfaction scale of 1 to 10, scores of zero are treated as missing or invalid codes and replaced with missing values.

The remaining recorded scores are retained without inventing values for missing responses. The limited range of observed scores is documented as a limitation of the dataset.

In [ ]:
date_col = [
    'appointment_datetime',
    'arrival_datetime',
    'discharge_datetime'
]
for col in date_col:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        dayfirst=True,
        errors="coerce"
    )
    df_clean[date_col].info()
    display(df_clean[date_col].head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1801 entries, 0 to 1800
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   appointment_datetime  1518 non-null   datetime64[ns]
 1   arrival_datetime      1691 non-null   object        
 2   discharge_datetime    1589 non-null   object        
dtypes: datetime64[ns](1), object(2)
memory usage: 42.3+ KB


,appointment_datetime,arrival_datetime,discharge_datetime
0,2025-10-16,16/10/2025,16/10/2025
1,2024-07-01,07/01/2024,NaN
2,2024-12-03,12/03/2024,12/03/2024
3,NaT,17/10/2025,17/10/2025
4,2025-08-20,20/08/2025,20/08/2025
5,2024-09-30,30/09/2024,30/09/2024
6,2025-05-09,05/09/2025,05/09/2025
7,2025-07-12,07/12/2025,NaN
8,NaT,07/08/2024,07/08/2024
9,2024-09-12,12/09/2024,12/09/2024


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1801 entries, 0 to 1800
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   appointment_datetime  1518 non-null   datetime64[ns]
 1   arrival_datetime      1691 non-null   datetime64[ns]
 2   discharge_datetime    1589 non-null   object        
dtypes: datetime64[ns](2), object(1)
memory usage: 42.3+ KB


,appointment_datetime,arrival_datetime,discharge_datetime
0,2025-10-16,2025-10-16,16/10/2025
1,2024-07-01,2024-01-07,NaN
2,2024-12-03,2024-03-12,12/03/2024
3,NaT,2025-10-17,17/10/2025
4,2025-08-20,2025-08-20,20/08/2025
5,2024-09-30,2024-09-30,30/09/2024
6,2025-05-09,2025-09-05,05/09/2025
7,2025-07-12,2025-12-07,NaN
8,NaT,2024-08-07,07/08/2024
9,2024-09-12,2024-09-12,12/09/2024


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1801 entries, 0 to 1800
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   appointment_datetime  1518 non-null   datetime64[ns]
 1   arrival_datetime      1691 non-null   datetime64[ns]
 2   discharge_datetime    1589 non-null   datetime64[ns]
dtypes: datetime64[ns](3)
memory usage: 42.3 KB


,appointment_datetime,arrival_datetime,discharge_datetime
0,2025-10-16,2025-10-16,2025-10-16
1,2024-07-01,2024-01-07,NaT
2,2024-12-03,2024-03-12,2024-03-12
3,NaT,2025-10-17,2025-10-17
4,2025-08-20,2025-08-20,2025-08-20
5,2024-09-30,2024-09-30,2024-09-30
6,2025-05-09,2025-09-05,2025-09-05
7,2025-07-12,2025-12-07,NaT
8,NaT,2024-08-07,2024-08-07
9,2024-09-12,2024-09-12,2024-09-12


## 7. Date and Time Conversion

The appointment, arrival, and discharge columns are converted into datetime format using pandas.

The `pd.to_datetime()` function converts date values into a format that Python can compare and analyse.

The `dayfirst=True` argument specifies day-first date interpretation. The `errors="coerce"` argument converts values that cannot be interpreted as valid dates into `NaT`, which represents a missing datetime value.

Correctly formatted dates allow comparisons between appointment, arrival, and discharge times.

In [ ]:
print(
    "Appointments after arrival:",
    (df_clean["appointment_datetime"] > df_clean["arrival_datetime"]).sum()
)

# Check for discharge before arrival
print(
    "Discharge before arrival:",
    (df_clean["discharge_datetime"] < df_clean["arrival_datetime"]).sum()
)

Appointments after arrival: 92
Discharge before arrival: 0


In [ ]:
# Display records where appointment time is after arrival
problem_dates = df_clean[
    (df_clean["appointment_datetime"] >
     df_clean["arrival_datetime"])
].copy()

display(
    problem_dates[
        [
            "appointment_id",
            "appointment_datetime",
            "arrival_datetime",
            "discharge_datetime"
        ]
    ].head(10)
)

print("Total records to review:", len(problem_dates))





,appointment_id,appointment_datetime,arrival_datetime,discharge_datetime
1,A00002,2024-07-01,2024-01-07,NaT
2,A00003,2024-12-03,2024-03-12,2024-03-12
34,A00035,2024-10-02,2024-02-10,2024-02-10
36,A00037,2025-09-07,2025-07-09,NaT
48,A00049,2024-03-01,2024-01-03,2024-01-03
126,A00128,2024-06-02,2024-02-06,2024-02-06
127,A00129,2025-12-04,2025-04-12,2025-04-12
152,A00154,2025-06-03,2025-03-06,2025-03-06
197,A00201,2025-12-08,2025-08-12,2025-08-12
225,A00229,2024-12-02,2024-02-12,2024-02-12


Total records to review: 92


## 8. Date Consistency Validation

The appointment, arrival, and discharge timestamps are compared to identify potential inconsistencies.

The analysis identified 92 records where the recorded appointment time was later than the recorded arrival time. No records were identified where discharge occurred before arrival.

These records are flagged for review rather than automatically deleted because the differences may relate to how appointment and arrival times were recorded.

Missing timestamps are also examined to understand the limitations of the dataset.

In [ ]:
# Create a data-quality flag
df_clean["date_issue"] = "No issue identified"

# Flag appointments recorded after arrival
issue = (
    df_clean["appointment_datetime"].notna()
    & df_clean["arrival_datetime"].notna()
    & (
        df_clean["appointment_datetime"]
        > df_clean["arrival_datetime"]
    )
)

df_clean.loc[issue, "date_issue"] = "Review: appointment after arrival"

# Check the results
print(df_clean["date_issue"].value_counts())

date_issue
No issue identified                  1709
Review: appointment after arrival      92
Name: count, dtype: int64


## 9. Flagging Potential Date Issues

A new column called `date_issue` is added to the dataset.

Records are initially assigned the label "No issue identified". Records with a recorded appointment time later than the arrival time are assigned the label "Review: appointment after arrival".

This flag makes it easier to identify and filter potential data-quality issues during further analysis.

The flagged records remain in the dataset so that useful information is not lost unnecessarily.

In [ ]:
missing_summary = pd.DataFrame({
    "missing count : " : df.isna().sum(),
    " percentage :" : (df.isna().sum()/len(df)*100).round(2)
})
display(missing_summary)

,missing count :,percentage :
appointment_id,0,0.00
patient_id,0,0.00
hospital,224,12.44
department,0,0.00
appointment_datetime,23,1.28
arrival_datetime,110,6.11
discharge_datetime,212,11.77
age,910,50.53
gender,349,19.38
appointment_status,201,11.16


In [ ]:
output = "healthcare_cleaned.csv"
df_clean.to_csv(output,index=False)
print("Dataset saved successfully!")
print("Rows and columns:", df_clean.shape)
print("File name:", output)

Dataset saved successfully!
Rows and columns: (1801, 19)
File name: healthcare_cleaned.csv


In [ ]:
from google.colab import files
files.download("healthcare_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Exporting the Cleaned Dataset

After the cleaning and validation steps, the prepared dataset is exported as a CSV file named `healthcare_appointments_cleaned.csv`.

The `to_csv()` function saves the DataFrame, and `index=False` prevents the DataFrame index from being added as an extra column.

The final dataset dimensions are checked to confirm the number of rows and columns.

The file is then downloaded from Google Colab so that it can be imported into Power BI for dashboard development.